# Load Documents for a RAG Pipeline

**Course author:** Jayneel Girnara

Explore LangChain loaders that convert local text, PDF, CSV, web pages, and Wikipedia results into `Document` objects for downstream retrieval.

## Learning goals
- Load several source types with their matching document loader.
- Inspect document text and metadata.
- Recognize which sources require local files, internet access, or an external API.

## Before you start
Install this repository's dependencies. Place the example files under `data/` as shown in the project structure. Web and Wikipedia examples need network access; some sources may rate-limit automated requests. Keep secrets in .env, never in notebook cells.

## How to work through it
Run one loader section at a time, top to bottom. Each section is independent after setup. Check the `data/` paths if a local loader reports that a file cannot be found.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

load_dotenv(project_root / ".env")


## 1. Load plain text

`TextLoader` returns a list of `Document` objects. Inspect its `page_content` before passing documents to a splitter.

A **document loader** converts a source file to a list of LangChain `Document` objects. The text is in `page_content`; `metadata` can retain the source path or other details. A loader reads and extracts—it does not embed or ask a model to answer.

**Checkpoint:** Where would you inspect the loaded text?

In [8]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader(project_root / "data" / "speech.txt")
texts=loader.load()
len(texts)


oneDoc=texts[0]
print(oneDoc.page_content)

“My dear friends, students, and respected farmers,

Today, I want to speak from the heart about three pillars that define the future of our nation:
our youth, our education system, and our farmers.

🎓 For the Youth

You are the energy that drives our state forward.
Our commitment is to give you:
	•	Skill-based education
	•	Digital learning opportunities
	•	Better employment pathways
	•	Support for innovation, startups, and entrepreneurship

Your dreams are not just your own—they are the dreams of our entire society.

📚 For Education

Education must not be just a formality; it must be a force of transformation.
We aim to:
	•	Upgrade government schools with smart classrooms
	•	Improve teacher training
	•	Expand scholarships for higher education
	•	Promote technical and vocational courses
	•	Ensure every child gets a fair opportunity to learn and grow

Education is the foundation on which we will build tomorrow.

👨‍🌾 For Farmers

Our farmers are our pride.
We honor their hard work and pro

## 2. Load a PDF

`PyPDFLoader` extracts page text and records page metadata. The sample path is relative to the repository root.

`PyPDFLoader` typically returns a document per page, with page/source metadata. PDF text extraction may not preserve layout or work for scanned images without OCR.

**Checkpoint:** Why is page metadata useful when answering questions from a PDF?

In [9]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader(project_root / "data" / "medical_report.pdf")
docs=loader.load()
len(docs)
oneDoc=docs[0]
print(oneDoc.page_content)

Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM
DR NITIN NAHAR
474764803
Ms. NIKITA  CHUDHARY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status 
10/7/2025  6:31:50PM
Final
Collected at            : Processed at             :BHOPAL CC-82
Mr Rachel V John Pata So Vitus John Mig 26 
Graund,Indrapuri, Phone: 8770817968
 
LPL - Bhopal Lab II
Plot No.05, Mandakini Housing Society, Near 
Apurti Shopping Mall, Kolar Main Road, 
Bhopal, M.P. -  462042
Test Report      
Test Name Results Units Bio. Ref. Interval
HEMOGRAM
(SLS Method, Sheath Flow DC Detection Method, Fluorescent Flow Cytometry, Calculated & Capillary Photometry)
Hemoglobin  12.00 - 15.00 g/dL13.10
Packed Cell Volume (PCV)  36.00 - 46.00 %39.50
RBC Count  3.80 - 4.80 mill/mm34.47
MCV  83.00 - 101.00 fL88.40
Mentzer Index   19.8
MCH  27.00 - 32.00 pg29.30
MCHC  31.50 - 34.50 g/dL33.20
Red Cell Distribution Width (RDW)  11.60 - 14.00 %11.90
Total Leukocyte Count (TLC)  

## 3. Load tabular data

`CSVLoader` turns each row into a document. Use only data that is safe to share with the model provider.

`CSVLoader` represents rows as documents, which is convenient for retrieval over records. Large or sensitive tables need deliberate filtering: loading every row can use substantial memory, and sending row text to a hosted embedding/model API may disclose that data.

**Checkpoint:** What should you decide before embedding rows from a CSV?

In [11]:
from langchain_community.document_loaders import CSVLoader
loader=CSVLoader(project_root / "data" / "employee_records.csv")
docs=loader.load()
len(docs)



30000

## 4. Load a web page

`WebBaseLoader` fetches a public URL; network access and site policies apply.

`WebBaseLoader` fetches a web page and extracts text; the site may change its markup, block automated requests, or restrict reuse. Network-based examples are less deterministic than local-file examples.

**Checkpoint:** Why can a successful loader today fail later?

In [16]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader(web_path="https://www.udemy.com/courses/search/?src=ukw&q=generative")
docs=loader.load()



## 5. Search Wikipedia

`WikipediaLoader` retrieves matching pages over the network. Results can vary over time, and the API may rate-limit requests. The following cell loads the matching pages and returns their count.

`WikipediaLoader` searches Wikipedia and returns matching page documents; it requires network access. Search results can vary, and the service may rate-limit requests. Inspect the returned `page_content` and metadata rather than assuming a query always yields the same pages.

**Checkpoint:** Which examples in this notebook need an internet connection?

In [27]:
from langchain_community.document_loaders import WikipediaLoader
loader=WikipediaLoader(query="generative artificial intelligence",load_max_docs=2)
docs=loader.load()
len(docs)
